# v10.1: The Pure Contextual Manifold

## Sparse Geometric Signal Transport — Built from First Principles

**This is not a modification of v9.** This notebook implements the architecture described in `README_PhD_Exploration_claude_v2.md` and `README_PhD_Exploration_Gemini.md` from scratch, without borrowing structure from any prior version.

### The Core Thesis

Versions 1-9 built an elegant mathematical engine — fiber bundles, gauge connections, Langevin-Hopfield dynamics — but drove it over a flat positional landscape. Each version compensated for a **context-blind manifold** by strapping on additional forces (causal convolution, lateral inhibition, subbundle attention). The result: a four-force Frankenstein SDE where the dominant force (Hopfield gradient) fought the others.

**The context is not a force applied to the settling token. The context IS the landscape itself.**

### The v10.1 Paradigm

1. **Contextual Manifold** (Wilson line as parallel scan): `q_t = A(x_t) * q_{t-1} + B(x_t) * psi(x_t)`
2. **Curvature-Aware Memory Bank**: high manifold curvature → broader attractor landscape → more exploration
3. **Pure Langevin SDE**: ONLY the Hopfield gradient + annealed noise. No causal conv. No attention. No lateral inhibition.
4. **Cooperative Manifold Construction**: each block refines `q_t` based on settled representations

### What's Removed (vs v9)

| Component | v9 | v10.1 | Rationale |
|---|---|---|---|
| Causal convolution | Force 2 in SDE | **Removed** | Band-aid for context-blind manifold |
| Lateral inhibition | Force 3 in SDE | **Removed** | Unnecessary with contextual routing |
| Per-subbundle attention | Force 4 in SDE | **Removed** | Attention builds the manifold, not a settling force |
| Per-step force scheduling | 12 learned params | **Removed** | Single force needs no scheduling |
| 4-force CausalScoreFunction | ~420K params/block | **Removed** | Replaced by pure Hopfield gradient |

### What's New

| Component | Description | Source |
|---|---|---|
| Contextual Manifold | Parallel scan Wilson line | Both READMEs, Sec 4.2 |
| Curvature-aware routing | k_t depends on manifold curvature | Claude README Sec 6.2 |
| Manifold refinement | Per-block cooperative geometry construction | Claude README Sec 3.1, Finding 5 |
| Curvature-modulated beta | Exploration breadth in high-curvature regions | Claude README Sec 6.5 |
| Final-step proximal sparsity | Threshold only at last Langevin step | Claude README Sec 8.3 |
| Multi-scale initialization | Fourier-inspired manifold retention rates | Claude README Sec 8.5 |

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from typing import Optional, Tuple, Dict, List
from dataclasses import dataclass
from collections import Counter
import math
import os
import urllib.request
from tqdm.notebook import tqdm

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

data_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "v8_real_text", "data")
data_path = os.path.join(data_dir, "input.txt")

if not os.path.exists(data_path):
    os.makedirs(data_dir, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r") as f:
    text = f.read()

def encode(s): return [ord(c) for c in s]
def decode(t): return "".join(chr(x) for x in t if 0 <= x < 256)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"Train: {len(train_data):,} | Val: {len(val_data):,} chars")

Device: cpu
Train: 1,003,854 | Val: 111,540 chars


In [6]:
@dataclass
class ArchitectureConfig:
    # v10.1: Pure Contextual Manifold
    # Removed: context_expand, attn_top_k, inhibition_gamma (forces 2-4 gone)
    # Added: k_wta_min/max (curvature-aware), curvature_beta_scale

    fiber_dim: int = 256
    n_subbundles: int = 8
    manifold_dim: int = 128

    vocab_size: int = 256
    max_seq_len: int = 128

    atoms_per_subbundle: int = 128
    k_wta_min: int = 8       # atoms at low curvature (decisive settling)
    k_wta_max: int = 24      # atoms at high curvature (broad exploration)

    n_blocks: int = 4
    langevin_steps: int = 6
    langevin_lr: float = 0.1
    beta_init: float = 1.0
    beta_final: float = 10.0

    sparsity_lambda: float = 0.01    # proximal threshold (final step only)
    curvature_beta_scale: float = 0.5  # curvature modulation of inverse temperature

    learning_rate: float = 3e-4
    dropout: float = 0.1
    batch_size: int = 16
    seq_len: int = 64
    max_steps: int = 10000
    eval_interval: int = 200
    eval_steps: int = 20

    @property
    def subbundle_dim(self):
        assert self.fiber_dim % self.n_subbundles == 0
        return self.fiber_dim // self.n_subbundles

config = ArchitectureConfig()
print(f"Fiber: {config.fiber_dim} = {config.n_subbundles} x {config.subbundle_dim}")
print(f"Manifold: {config.manifold_dim}d (contextual, parallel scan)")
print(f"Memory: {config.atoms_per_subbundle} atoms/sub, k in [{config.k_wta_min}, {config.k_wta_max}] (curvature-aware)")
print(f"Langevin: {config.langevin_steps} steps, PURE Hopfield gradient (no other forces)")
print(f"Blocks: {config.n_blocks} with per-block manifold refinement")

def get_batch(split_data, cfg):
    max_start = len(split_data) - cfg.seq_len - 1
    starts = torch.randint(0, max_start, (cfg.batch_size,))
    return torch.stack([split_data[s:s+cfg.seq_len] for s in starts]).to(device)

Fiber: 256 = 8 x 32
Manifold: 128d (contextual, parallel scan)
Memory: 128 atoms/sub, k in [8, 24] (curvature-aware)
Langevin: 6 steps, PURE Hopfield gradient (no other forces)
Blocks: 4 with per-block manifold refinement


## Architecture

### Contextual Manifold (Wilson Line as Parallel Scan)
The Wilson line from gauge theory IS the selective state space model recurrence:
`q_t = A(x_t) * q_{t-1} + B(x_t) * psi(x_t)`. A_proj bias is initialized with a range
of values for multi-scale retention (some dimensions decay fast for local context,
others retain for long-range dependencies — connecting to Fourier structure of counting
manifolds from Gurnee et al.).

### Curvature-Aware Memory Bank
Discrete curvature `kappa_t = ||q_t - 2*q_{t-1} + q_{t-2}|| / ||q_t - q_{t-1}||^2`.
High curvature (rapid context change) → more atoms active → broader attractor landscape.
Low curvature → fewer atoms → decisive settling. Memory atoms are the architectural
analog of Gurnee et al.'s place cells — more should tile where the manifold curves sharply.

### Pure Langevin Descent
Single force: Hopfield gradient. `dx = -grad_x E_q(x; M_q_t) dt + sqrt(2/beta_t) dW`.
Beta is curvature-modulated: high curvature → lower effective beta → more exploration.
Proximal sparsity (Axiom 5) applied only at the final Langevin step.

### Manifold Refiner
Each block refines q_t based on settled representations (cooperative geometry construction
from Gurnee et al. Finding 5). The manifold evolves through blocks: coarse → refined.

In [7]:
def linear_scan(gates, inputs):
    """Sequential associative scan: h_t = gates_t * h_{t-1} + inputs_t.
    Replace with Blelloch scan or mamba_ssm.selective_scan_fn for GPU parallelism.
    """
    B, T, D = gates.shape
    h_list = [inputs[:, 0]]
    for t in range(1, T):
        h_list.append(gates[:, t] * h_list[-1] + inputs[:, t])
    return torch.stack(h_list, dim=1)


class ContextualManifold(nn.Module):
    """Wilson line as selective SSM. q_t = f(x_0, ..., x_t) via content-dependent recurrence."""

    def __init__(self, cfg):
        super().__init__()
        self.A_proj = nn.Linear(cfg.fiber_dim, cfg.manifold_dim)
        self.B_proj = nn.Linear(cfg.fiber_dim, cfg.manifold_dim)
        self.psi_proj = nn.Linear(cfg.fiber_dim, cfg.manifold_dim)
        self.norm = nn.LayerNorm(cfg.manifold_dim)

        # Multi-scale retention: range of decay rates across manifold dimensions
        with torch.no_grad():
            nn.init.uniform_(self.A_proj.bias, -2.0, 2.0)

    def forward(self, x_sparse):
        A = torch.sigmoid(self.A_proj(x_sparse))
        B = torch.sigmoid(self.B_proj(x_sparse))
        psi = self.psi_proj(x_sparse)
        q = linear_scan(A, B * psi)
        return self.norm(q)


class SparseTokenEmbedding(nn.Module):
    """Sparse fiber bundle sections (Axiom 1) + contextual manifold coordinates."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.fiber_dim)
        self.topk_per_sub = max(1, cfg.subbundle_dim // 4)
        self.manifold = ContextualManifold(cfg)

    def forward(self, token_ids):
        B, T = token_ids.shape
        x = self.embedding(token_ids)

        chunks = x.chunk(self.cfg.n_subbundles, dim=-1)
        sparse_chunks = []
        for c in chunks:
            _, idx = torch.topk(c.abs(), self.topk_per_sub, dim=-1)
            mask = torch.zeros_like(c)
            mask.scatter_(-1, idx, 1.0)
            sparse_chunks.append(c * mask)
        x_sparse = torch.cat(sparse_chunks, dim=-1)

        q = self.manifold(x_sparse)
        return x_sparse, q


class CurvatureAwareMemoryBank(nn.Module):
    """Memory bank with curvature-dependent atom selection (Axiom 3 + Sec 6.2).

    High curvature -> more atoms active -> broader attractor landscape.
    Low curvature -> fewer atoms -> decisive settling.
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        sd, K, A = cfg.subbundle_dim, cfg.n_subbundles, cfg.atoms_per_subbundle

        self.dictionaries = nn.ParameterList(
            [nn.Parameter(torch.randn(A, sd) * 0.02) for _ in range(K)]
        )
        self.routers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.manifold_dim + sd, A), nn.SiLU(), nn.Linear(A, A)
            ) for _ in range(K)
        ])

    def compute_curvature(self, q):
        """Discrete curvature of the manifold trajectory.
        kappa_t = ||accel_t|| / ||vel_t||^2
        """
        B, T, D = q.shape
        if T < 3:
            return torch.full((B, T), 0.5, device=q.device)

        vel = q[:, 1:] - q[:, :-1]
        accel = q[:, 2:] - 2 * q[:, 1:-1] + q[:, :-2]
        accel_norm = accel.norm(dim=-1)
        vel_norm_sq = vel[:, :-1].norm(dim=-1).pow(2).clamp(min=1e-8)
        kappa = accel_norm / vel_norm_sq

        # Pad first two positions with the first computed curvature
        pad_val = kappa[:, :1]
        kappa = torch.cat([pad_val, pad_val, kappa], dim=1)
        return kappa

    def forward(self, q, x):
        cfg = self.cfg
        sd = cfg.subbundle_dim
        B, T = q.shape[0], q.shape[1]

        curvature = self.compute_curvature(q)

        # Normalize curvature to [0, 1] for atom count interpolation
        kappa_norm = (curvature / (curvature.max().clamp(min=1e-8))).clamp(0, 1)

        # Curvature-dependent k per token
        k_per_token = (cfg.k_wta_min + kappa_norm * (cfg.k_wta_max - cfg.k_wta_min)).long()
        k_per_token = k_per_token.clamp(cfg.k_wta_min, cfg.k_wta_max)

        q_flat = q.reshape(B * T, -1)
        x_flat = x.reshape(B * T, -1)
        k_flat = k_per_token.reshape(B * T)

        # Mask: 1 for atom indices < k_t, 0 beyond (zeroes out excess atoms)
        atom_mask = torch.arange(cfg.k_wta_max, device=q.device).unsqueeze(0) < k_flat.unsqueeze(-1)

        x_chunks = x_flat.split(sd, dim=-1)
        M_list = []

        for chunk, dic, router in zip(x_chunks, self.dictionaries, self.routers):
            D_n = F.normalize(dic, dim=-1)
            logits = router(torch.cat([q_flat, chunk], dim=-1))
            _, idx = torch.topk(logits, cfg.k_wta_max, dim=-1)
            atoms = D_n[idx]
            atoms = atoms * atom_mask.unsqueeze(-1).float()
            M_list.append(atoms)

        return M_list, curvature


class PureLangevinDescent(nn.Module):
    """Annealed Langevin dynamics with pure Hopfield gradient (Axiom 4).
    Single force. The context IS the landscape.
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

    def hopfield_gradient(self, x_chunk, M_k, beta):
        """Per-subbundle: -grad_x E = -sum_j softmax(beta * x^T m_j) m_j"""
        sim = torch.bmm(M_k, x_chunk.unsqueeze(-1)).squeeze(-1)
        if isinstance(beta, torch.Tensor):
            w = F.softmax(beta.unsqueeze(-1) * sim, dim=-1)
        else:
            w = F.softmax(beta * sim, dim=-1)
        return -torch.bmm(w.unsqueeze(1), M_k).squeeze(1)

    def forward(self, x_seq, M_q_all, curvature=None,
                return_trajectory=False, return_diagnostics=False):
        cfg = self.cfg
        B, T, D = x_seq.shape
        sd = cfg.subbundle_dim
        x = x_seq.clone()

        betas = torch.linspace(cfg.beta_init, cfg.beta_final, cfg.langevin_steps)
        trajectory = [x.detach().clone()] if return_trajectory else None
        diagnostics = [] if return_diagnostics else None

        for step in range(cfg.langevin_steps):
            beta_base = betas[step].item()

            # Curvature-modulated inverse temperature
            if curvature is not None:
                kappa = curvature.clamp(0, 5.0)
                beta_eff = beta_base / (1.0 + cfg.curvature_beta_scale * kappa)
                beta_flat = beta_eff.reshape(B * T)
                noise_scale = torch.sqrt(
                    2.0 * cfg.langevin_lr / beta_eff.clamp(min=0.1)
                ).unsqueeze(-1)
                noise = noise_scale * torch.randn_like(x)
            else:
                beta_flat = beta_base
                noise = math.sqrt(2.0 * cfg.langevin_lr / beta_base) * torch.randn_like(x)

            x_flat = x.reshape(B * T, D)
            x_chunks = x_flat.split(sd, dim=-1)
            grad_chunks = [
                self.hopfield_gradient(xc, mk, beta_flat)
                for xc, mk in zip(x_chunks, M_q_all)
            ]
            grad_E = torch.cat(grad_chunks, dim=-1).reshape(B, T, D)

            x = x - cfg.langevin_lr * grad_E + noise

            # Proximal sparsity only on final step (Axiom 5, Sec 8.3)
            if step == cfg.langevin_steps - 1 and cfg.sparsity_lambda > 0:
                threshold = cfg.sparsity_lambda * cfg.langevin_lr
                x = torch.sign(x) * F.relu(x.abs() - threshold)

            if return_trajectory:
                trajectory.append(x.detach().clone())
            if return_diagnostics:
                diagnostics.append({
                    'grad_norm': grad_E.detach().norm(dim=-1).mean().item(),
                    'state_norm': x.detach().norm(dim=-1).mean().item(),
                    'beta_mean': beta_flat.mean().item() if isinstance(beta_flat, torch.Tensor) else beta_flat,
                })

        result = {'settled': x}
        if return_trajectory:
            result['trajectory'] = trajectory
        if return_diagnostics:
            result['diagnostics'] = diagnostics
        return result


class ManifoldRefiner(nn.Module):
    """Per-block manifold evolution (cooperative geometry construction, Gurnee et al. Finding 5).
    q' = q + gate(q, delta) * delta, where delta is derived from settled representations.
    """

    def __init__(self, cfg):
        super().__init__()
        self.refine = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.manifold_dim),
            nn.SiLU(),
            nn.Linear(cfg.manifold_dim, cfg.manifold_dim),
        )
        self.gate = nn.Sequential(
            nn.Linear(cfg.manifold_dim * 2, cfg.manifold_dim),
            nn.Sigmoid()
        )

    def forward(self, q, x_settled):
        delta = self.refine(x_settled)
        g = self.gate(torch.cat([q, delta], dim=-1))
        return q + g * delta


class PureContextualBlock(nn.Module):
    """One block: curvature-aware memory bank -> pure Langevin -> manifold refinement -> residual.
    No causal conv. No attention. No lateral inhibition.
    """

    def __init__(self, cfg):
        super().__init__()
        self.memory_bank = CurvatureAwareMemoryBank(cfg)
        self.langevin = PureLangevinDescent(cfg)
        self.refiner = ManifoldRefiner(cfg)
        self.norm = nn.LayerNorm(cfg.fiber_dim)
        self.dropout = nn.Dropout(cfg.dropout)
        self.residual_gate = nn.Sequential(
            nn.Linear(cfg.fiber_dim * 2, cfg.fiber_dim), nn.Sigmoid()
        )

    def forward(self, x_seq, q, return_diagnostics=False):
        B, T, D = x_seq.shape

        M_q_all, curvature = self.memory_bank(q, x_seq)

        result = self.langevin(
            x_seq, M_q_all, curvature=curvature,
            return_diagnostics=return_diagnostics
        )
        x_settled = result['settled']

        gate_in = torch.cat([x_settled, x_seq], dim=-1)
        gate = self.residual_gate(gate_in.reshape(-1, D * 2)).reshape(B, T, D)
        x_out = self.norm(gate * self.dropout(x_settled) + (1 - gate) * x_seq)

        q_refined = self.refiner(q, x_out)

        output = {'x': x_out, 'q': q_refined, 'curvature': curvature}
        if return_diagnostics:
            output['langevin_diagnostics'] = result.get('diagnostics', [])
        return output


class ContextualManifoldCLM(nn.Module):
    """v10.1: Pure Contextual Manifold CLM.

    The context is not a force applied to the settling token.
    The context IS the landscape itself.

    The manifold evolves through blocks (cooperative geometry construction).
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = SparseTokenEmbedding(cfg)
        self.blocks = nn.ModuleList([PureContextualBlock(cfg) for _ in range(cfg.n_blocks)])
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim), nn.SiLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.fiber_dim, cfg.vocab_size),
        )
        self.register_buffer("block_loss_weights", torch.linspace(0.1, 1.0, cfg.n_blocks))

    def forward(self, token_ids, return_manifold=False, return_diagnostics=False):
        B, T = token_ids.shape
        x, q = self.embedding(token_ids)

        intermediate_logits = []
        curvatures = []
        block_diagnostics = []

        for block in self.blocks:
            result = block(x, q, return_diagnostics=return_diagnostics)
            x = result['x']
            q = result['q']
            curvatures.append(result['curvature'].detach())
            intermediate_logits.append(self.decoder(x)[:, :-1, :])
            if return_diagnostics:
                block_diagnostics.append(result.get('langevin_diagnostics', []))

        info = {
            "mean_sparsity": (x.abs() < 1e-3).float().mean().item(),
            "intermediate_logits": intermediate_logits,
            "block_loss_weights": self.block_loss_weights,
            "curvatures": curvatures,
        }
        if return_manifold:
            info["manifold_coords"] = q
            info["final_repr"] = x
        if return_diagnostics:
            info["block_diagnostics"] = block_diagnostics

        return intermediate_logits[-1], info


model = ContextualManifoldCLM(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_manifold = sum(p.numel() for p in model.embedding.manifold.parameters())
n_memory = sum(
    sum(p.numel() for p in blk.memory_bank.parameters())
    for blk in model.blocks
)
n_refiner = sum(
    sum(p.numel() for p in blk.refiner.parameters())
    for blk in model.blocks
)

print(f"Total parameters: {n_params:,}")
print(f"  Contextual manifold: {n_manifold:,} ({100*n_manifold/n_params:.1f}%)")
print(f"  Memory banks (4 blk): {n_memory:,} ({100*n_memory/n_params:.1f}%)")
print(f"  Manifold refiners:    {n_refiner:,} ({100*n_refiner/n_params:.1f}%)")
print(f"  Attention params:     0 (REMOVED)")
print(f"  Causal conv params:   0 (REMOVED)")
print(f"  Inhibition params:    0 (REMOVED)")
print(f"\n{config.n_blocks} blocks x {config.langevin_steps} Langevin steps")
print(f"Each step: ONLY Hopfield gradient (pure SDE)")

Total parameters: 2,471,552
  Contextual manifold: 98,944 (4.0%)
  Memory banks (4 blk): 1,318,912 (53.4%)
  Manifold refiners:    329,216 (13.3%)
  Attention params:     0 (REMOVED)
  Causal conv params:   0 (REMOVED)
  Inhibition params:    0 (REMOVED)

4 blocks x 6 Langevin steps
Each step: ONLY Hopfield gradient (pure SDE)


## Training

Novel tracked diagnostics (not present in v9):
- **Curvature statistics**: mean/max manifold curvature per eval
- **Manifold variance**: how much q_t varies across positions (manifold utilization)

Loss = deep supervision CE + dictionary coherence regularizer

In [8]:
@torch.no_grad()
def estimate_loss(model, cfg):
    model.eval()
    results = {}
    for name, sd in [("train", train_data), ("val", val_data)]:
        tot_ce, tot_ok, tot_n, tot_sp = 0., 0, 0, 0.
        bces = [0.] * cfg.n_blocks
        curvature_means, curvature_maxs, manifold_vars = [], [], []

        for _ in range(cfg.eval_steps):
            b = get_batch(sd, cfg)
            logits, info = model(b, return_manifold=True)
            tgt = b[:, 1:]
            ce = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), tgt.reshape(-1))
            tot_ce += ce.item()
            tot_ok += (logits.argmax(-1) == tgt).sum().item()
            tot_n += tgt.numel()
            tot_sp += info["mean_sparsity"]
            for i, bl in enumerate(info["intermediate_logits"]):
                bces[i] += F.cross_entropy(bl.reshape(-1, cfg.vocab_size), tgt.reshape(-1)).item()
            if info["curvatures"]:
                last_curv = info["curvatures"][-1]
                curvature_means.append(last_curv.mean().item())
                curvature_maxs.append(last_curv.max().item())
            if "manifold_coords" in info:
                manifold_vars.append(info["manifold_coords"].var(dim=1).mean().item())

        n = cfg.eval_steps
        results[name] = {
            "ce": tot_ce/n, "acc": tot_ok/tot_n, "sparsity": tot_sp/n,
            "block_ces": [c/n for c in bces],
            "curvature_mean": np.mean(curvature_means) if curvature_means else 0,
            "curvature_max": np.mean(curvature_maxs) if curvature_maxs else 0,
            "manifold_var": np.mean(manifold_vars) if manifold_vars else 0,
        }
    model.train()
    return results


optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.01)

history = {
    "step": [], "train_ce": [], "val_ce": [], "train_acc": [], "val_acc": [],
    "train_sparsity": [], "val_sparsity": [],
    "train_bpc": [], "val_bpc": [],
    "block_val_ces": [[] for _ in range(config.n_blocks)],
    "curvature_mean": [], "curvature_max": [],
    "manifold_var": [],
}

print("Training v10.1 (pure contextual manifold) on Tiny Shakespeare")
print(f"Steps: {config.max_steps}, Batch: {config.batch_size}, Seq: {config.seq_len}")
print("Pure Langevin SDE: ONLY Hopfield gradient. No other forces.")
print(f"v9 baseline to beat: Val BPC 2.65 | Val Acc 45%")
print("=" * 70)

model.train()
pbar = tqdm(range(config.max_steps), desc="Training", unit="step")

for step in pbar:
    if step % config.eval_interval == 0:
        res = estimate_loss(model, config)
        tr, vl = res["train"], res["val"]
        history["step"].append(step)
        for k in ["ce", "acc", "sparsity"]:
            history[f"train_{k}"].append(tr[k])
            history[f"val_{k}"].append(vl[k])
        history["train_bpc"].append(tr["ce"] / math.log(2))
        history["val_bpc"].append(vl["ce"] / math.log(2))
        history["curvature_mean"].append(vl["curvature_mean"])
        history["curvature_max"].append(vl["curvature_max"])
        history["manifold_var"].append(vl["manifold_var"])
        for i in range(config.n_blocks):
            history["block_val_ces"][i].append(vl["block_ces"][i])
        tqdm.write(
            f"Step {step:5d} | Train CE: {tr['ce']:.3f} | Val CE: {vl['ce']:.3f} | "
            f"Val BPC: {vl['ce']/math.log(2):.2f} | Val Acc: {vl['acc']:.1%} | "
            f"Sp: {vl['sparsity']:.1%} | kappa: {vl['curvature_mean']:.2f}"
        )

    batch = get_batch(train_data, config)
    optimizer.zero_grad()
    logits, info = model(batch)
    targets = batch[:, 1:]

    # Deep supervision
    ce_loss = sum(
        w * F.cross_entropy(bl.reshape(-1, config.vocab_size), targets.reshape(-1))
        for bl, w in zip(info["intermediate_logits"], info["block_loss_weights"])
    ) / info["block_loss_weights"].sum()

    # Dictionary coherence regularizer
    dcl, nd = 0., 0
    for blk in model.blocks:
        for d in blk.memory_bank.dictionaries:
            Dn = F.normalize(d, dim=-1)
            g = Dn @ Dn.T
            dcl += (g - torch.eye(g.size(0), device=g.device)).pow(2).mean()
            nd += 1
    dcl /= max(nd, 1)

    loss = ce_loss + 0.1 * dcl
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

# Final eval
res = estimate_loss(model, config)
tr, vl = res["train"], res["val"]
history["step"].append(config.max_steps)
for k in ["ce", "acc", "sparsity"]:
    history[f"train_{k}"].append(tr[k])
    history[f"val_{k}"].append(vl[k])
history["train_bpc"].append(tr["ce"] / math.log(2))
history["val_bpc"].append(vl["ce"] / math.log(2))
history["curvature_mean"].append(vl["curvature_mean"])
history["curvature_max"].append(vl["curvature_max"])
history["manifold_var"].append(vl["manifold_var"])
for i in range(config.n_blocks):
    history["block_val_ces"][i].append(vl["block_ces"][i])

print("=" * 70)
print(f"FINAL | Val CE: {vl['ce']:.3f} | Val BPC: {vl['ce']/math.log(2):.2f} | "
      f"Val Acc: {vl['acc']:.1%} | Val PPL: {math.exp(vl['ce']):.1f}")
print(f"v9:    Val BPC 2.65 | Val Acc 45% | Val PPL 6.3")

Training v10.1 (pure contextual manifold) on Tiny Shakespeare
Steps: 10000, Batch: 16, Seq: 64
Pure Langevin SDE: ONLY Hopfield gradient. No other forces.
v9 baseline to beat: Val BPC 2.65 | Val Acc 45%


Training:   0%|          | 0/10000 [00:00<?, ?step/s]

Step     0 | Train CE: 5.563 | Val CE: 5.565 | Val BPC: 8.03 | Val Acc: 0.2% | Sp: 0.1% | kappa: 0.19
Step   200 | Train CE: 2.531 | Val CE: 2.558 | Val BPC: 3.69 | Val Acc: 27.7% | Sp: 0.1% | kappa: 0.06
Step   400 | Train CE: 2.444 | Val CE: 2.449 | Val BPC: 3.53 | Val Acc: 29.5% | Sp: 0.2% | kappa: 0.05
Step   600 | Train CE: 2.386 | Val CE: 2.415 | Val BPC: 3.48 | Val Acc: 30.1% | Sp: 0.2% | kappa: 0.04
Step   800 | Train CE: 2.383 | Val CE: 2.392 | Val BPC: 3.45 | Val Acc: 30.8% | Sp: 0.2% | kappa: 0.03
Step  1000 | Train CE: 2.375 | Val CE: 2.397 | Val BPC: 3.46 | Val Acc: 30.3% | Sp: 0.2% | kappa: 0.03
Step  1200 | Train CE: 2.371 | Val CE: 2.406 | Val BPC: 3.47 | Val Acc: 30.0% | Sp: 0.2% | kappa: 0.03


KeyboardInterrupt: 

In [ ]:
steps = history["step"]
fig, axes = plt.subplots(3, 3, figsize=(18, 15))

# Row 1: Performance
axes[0,0].plot(steps, history["train_ce"], 'b-', label='Train')
axes[0,0].plot(steps, history["val_ce"], 'r-', label='Val')
axes[0,0].set_title('Cross-Entropy'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(steps, history["train_bpc"], 'b-', label='Train')
axes[0,1].plot(steps, history["val_bpc"], 'r-', label='Val')
axes[0,1].axhline(y=2.65, color='gray', linestyle='--', alpha=0.7, label='v9 baseline')
axes[0,1].set_title('Bits per Character'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(steps, history["train_acc"], 'b-', label='Train')
axes[0,2].plot(steps, history["val_acc"], 'r-', label='Val')
axes[0,2].axhline(y=0.45, color='gray', linestyle='--', alpha=0.7, label='v9 ceiling')
axes[0,2].set_title('Accuracy'); axes[0,2].legend(); axes[0,2].grid(True, alpha=0.3)

# Row 2: Training quality
gap = [v - t for v, t in zip(history["val_ce"], history["train_ce"])]
axes[1,0].plot(steps, gap, 'purple', lw=2)
axes[1,0].set_title('Generalization Gap (Val - Train CE)'); axes[1,0].grid(True, alpha=0.3)

colors = plt.cm.viridis(np.linspace(0.2, 0.9, config.n_blocks))
for i in range(config.n_blocks):
    axes[1,1].plot(steps, history["block_val_ces"][i], color=colors[i], label=f'Block {i}')
axes[1,1].set_title('Per-Block Val CE (deep supervision)')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

axes[1,2].plot(steps, history["val_sparsity"], 'r-')
axes[1,2].set_title('Output Sparsity (final-step proximal)')
axes[1,2].grid(True, alpha=0.3)

# Row 3: MANIFOLD GEOMETRY (novel — not in v9)
axes[2,0].plot(steps, history["curvature_mean"], 'teal', lw=2, label='Mean')
axes[2,0].plot(steps, history["curvature_max"], 'coral', lw=1, alpha=0.7, label='Max')
axes[2,0].set_title('Manifold Curvature Evolution')
axes[2,0].set_xlabel('Step'); axes[2,0].legend(); axes[2,0].grid(True, alpha=0.3)

axes[2,1].plot(steps, history["manifold_var"], 'steelblue', lw=2)
axes[2,1].set_title('Manifold Coordinate Variance')
axes[2,1].set_xlabel('Step'); axes[2,1].grid(True, alpha=0.3)

with torch.no_grad():
    model.eval()
    batch = get_batch(val_data, config)
    x, q = model.embedding(batch)
    M_list, _ = model.blocks[0].memory_bank(q, x)
    atom_counts = []
    for M_k in M_list:
        norms = M_k.norm(dim=-1)
        active = (norms > 1e-6).float().mean().item()
        atom_counts.append(active)
    model.train()

axes[2,2].bar(range(config.n_subbundles), atom_counts, color='teal', alpha=0.8)
axes[2,2].set_xlabel('Subbundle'); axes[2,2].set_ylabel('Active Fraction')
axes[2,2].set_title('Dictionary Atom Utilization (Block 0)')
axes[2,2].set_ylim(0, 1.1); axes[2,2].grid(True, alpha=0.3, axis='y')

plt.suptitle('v10.1: Pure Contextual Manifold — Training Diagnostics', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nFinal: Val BPC {history['val_bpc'][-1]:.2f} | Val Acc {history['val_acc'][-1]:.1%}")
print(f"v9:    Val BPC 2.65 | Val Acc 45%")
delta_bpc = history['val_bpc'][-1] - 2.65
delta_acc = (history['val_acc'][-1] - 0.45) * 100
print(f"Delta: BPC {delta_bpc:+.2f} | Acc {delta_acc:+.1f}pp")
print(f"\nManifold curvature (final): mean={history['curvature_mean'][-1]:.3f}, max={history['curvature_max'][-1]:.3f}")
print(f"Manifold variance (final): {history['manifold_var'][-1]:.4f}")

In [ ]:
@torch.no_grad()
def generate_text(model, prompt_str, max_new=200, temperature=0.8):
    model.eval()
    cfg = model.cfg
    ids = torch.tensor(encode(prompt_str), dtype=torch.long).unsqueeze(0).to(device)
    for _ in range(max_new):
        ctx = ids[:, -cfg.max_seq_len:] if ids.size(1) >= cfg.max_seq_len else ids
        logits, _ = model(ctx)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        ids = torch.cat([ids, torch.multinomial(probs, 1)], dim=1)
    return decode(ids[0].tolist())

print("=" * 60)
print("TEXT GENERATION (v10.1 pure contextual manifold, temp=0.8)")
print("=" * 60)
for p in ["ROMEO:\n", "To be or not to ", "The king ", "O, "]:
    print(f"\n--- Prompt: {repr(p)} ---")
    print(generate_text(model, p))

## PhD Diagnostic Suite

Ten tests to validate the contextual manifold hypothesis and characterize the pure SDE paradigm.
Tests 1-5 and 8-10 follow the PhD exploration documents. **Tests 6-7 are entirely new** —
the original force cooperation tests are irrelevant when there's only one force.

### Manifold Contextuality (Tests 1-3)
- **Test 1**: Same position, different contexts — manifold divergence (should be >> 0)
- **Test 2**: Manifold trajectory visualization — curved path reflecting content (not monotone)
- **Test 3**: Memory bank atom overlap — Jaccard similarity (should be < 1.0)

### Context Accumulation (Tests 4-5)
- **Test 4**: Linear probes on q_t — can we decode history from manifold coordinates?
- **Test 5**: Ablation — contextual vs positional manifold at test time

### Pure SDE Analysis (Tests 6-7, NEW)
- **Test 6**: Context-sensitive Hopfield gradient — does gradient direction change with context?
- **Test 7**: Curvature-dependent exploration — does high curvature activate more atoms?

### Gauge Curvature (Test 8)
- **Test 8**: Small-loop holonomy — nonzero = path-dependent transport

### Sparse Fiber Bundle Patterns (Tests 9-10)
- **Test 9**: Activation pattern divergence under context variation
- **Test 10**: Subbundle activation entropy (combinatorial capacity utilization)

In [ ]:
@torch.no_grad()
def test_manifold_contextuality(model, cfg):
    """Tests 1-3: Is the manifold actually contextual?"""
    model.eval()

    text_a = "ROMEO:\nO, she doth teach the torches to burn bright!"
    text_b = "JULIET:\nWhat's in a name? That which we call a rose"

    T = min(cfg.seq_len, len(text_a), len(text_b))
    ids_a = torch.tensor(encode(text_a[:T]), dtype=torch.long).unsqueeze(0).to(device)
    ids_b = torch.tensor(encode(text_b[:T]), dtype=torch.long).unsqueeze(0).to(device)

    _, info_a = model(ids_a, return_manifold=True)
    _, info_b = model(ids_b, return_manifold=True)
    q_a = info_a["manifold_coords"][0].cpu()
    q_b = info_b["manifold_coords"][0].cpu()

    # Test 1: Position-wise manifold divergence
    divergence = (q_a - q_b).norm(dim=-1).numpy()

    # Test 2: PCA trajectory visualization
    combined = torch.cat([q_a, q_b], dim=0)
    centered = combined - combined.mean(dim=0, keepdim=True)
    U, S, Vh = torch.linalg.svd(centered, full_matrices=False)
    explained = (S[:3] ** 2) / (S ** 2).sum()
    proj = (centered @ Vh.T[:, :3]).numpy()
    q_a_3d, q_b_3d = proj[:T], proj[T:]

    # Test 3: Memory bank atom overlap (Jaccard)
    x_a, q_a_t = model.embedding(ids_a)
    x_b, q_b_t = model.embedding(ids_b)
    block = model.blocks[0]

    jaccards = []
    for t in range(T):
        sub_jacs = []
        for k, router in enumerate(block.memory_bank.routers):
            sd = cfg.subbundle_dim
            chunk_a = x_a[0, t:t+1, k*sd:(k+1)*sd]
            chunk_b = x_b[0, t:t+1, k*sd:(k+1)*sd]
            q_at = q_a_t[0, t:t+1]
            q_bt = q_b_t[0, t:t+1]
            logits_a = router(torch.cat([q_at, chunk_a], dim=-1))
            logits_b = router(torch.cat([q_bt, chunk_b], dim=-1))
            _, idx_a = torch.topk(logits_a, cfg.k_wta_max, dim=-1)
            _, idx_b = torch.topk(logits_b, cfg.k_wta_max, dim=-1)
            set_a = set(idx_a[0].cpu().numpy().tolist())
            set_b = set(idx_b[0].cpu().numpy().tolist())
            jac = len(set_a & set_b) / len(set_a | set_b) if set_a | set_b else 1.0
            sub_jacs.append(jac)
        jaccards.append(np.mean(sub_jacs))

    fig = plt.figure(figsize=(18, 5))

    ax1 = fig.add_subplot(131)
    ax1.plot(divergence, 'b-', lw=2)
    ax1.set_xlabel('Position'); ax1.set_ylabel('||q_a - q_b||')
    ax1.set_title('Test 1: Manifold Divergence\n(same position, different contexts)')
    ax1.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='v1-v9 (always 0)')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(132, projection='3d')
    ax2.plot(*q_a_3d.T, 'b-o', ms=3, label='ROMEO...')
    ax2.plot(*q_b_3d.T, 'r-o', ms=3, label='JULIET...')
    ax2.set_title(f'Test 2: Manifold Trajectories (PCA)\n{explained.sum():.0%} variance in 3D')
    ax2.legend(fontsize=8)

    ax3 = fig.add_subplot(133)
    ax3.plot(jaccards, 'g-', lw=2)
    ax3.set_xlabel('Position'); ax3.set_ylabel('Jaccard Similarity')
    ax3.set_title('Test 3: Memory Bank Atom Overlap\n(1.0 = identical routing)')
    ax3.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='v1-v9 (always 1.0)')
    ax3.set_ylim(0, 1.1); ax3.legend(); ax3.grid(True, alpha=0.3)

    plt.suptitle('PhD Diagnostics: Manifold Contextuality (Tests 1-3)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"\nTest 1: Mean divergence = {divergence.mean():.4f} (v1-v9: 0.0000)")
    print(f"Test 3: Mean Jaccard   = {np.mean(jaccards):.4f} (v1-v9: 1.0000)")
    verdict = "PASS" if divergence.mean() > 0.01 else "FAIL"
    print(f"Verdict: {verdict}")

test_manifold_contextuality(model, config)

In [ ]:
@torch.no_grad()
def test_probing_and_ablation(model, cfg):
    """Tests 4-5: Context accumulation quality."""
    model.eval()

    # Test 4: Linear probe on q_t to predict previous character
    n_samples = 2000
    qs, targets = [], []
    for _ in range(n_samples // (cfg.batch_size * cfg.seq_len) + 1):
        batch = get_batch(val_data, cfg)
        _, info = model(batch, return_manifold=True)
        q = info["manifold_coords"][:, 1:, :]
        tgt = batch[:, :-1]
        qs.append(q.reshape(-1, cfg.manifold_dim).cpu())
        targets.append(tgt.reshape(-1).cpu())

    Q = torch.cat(qs, dim=0)[:n_samples]
    Y = torch.cat(targets, dim=0)[:n_samples]

    counts = torch.bincount(Y, minlength=256)
    top_chars = counts.topk(30).indices
    mask = torch.zeros(len(Y), dtype=torch.bool)
    for c in top_chars:
        mask |= (Y == c)
    Q_sub, Y_sub = Q[mask], Y[mask]

    n_train = int(0.8 * len(Q_sub))
    Q_train, Q_test = Q_sub[:n_train], Q_sub[n_train:]
    Y_train, Y_test = Y_sub[:n_train], Y_sub[n_train:]

    Y_onehot = F.one_hot(Y_train.long(), num_classes=256).float()
    W_probe = torch.linalg.lstsq(Q_train, Y_onehot).solution
    preds = (Q_test @ W_probe).argmax(dim=-1)
    probe_acc = (preds == Y_test).float().mean().item()
    baseline_acc = torch.bincount(Y_test.long(), minlength=256).max().item() / len(Y_test)

    # Test 5: Ablation — contextual vs positional manifold
    contextual_acc, n_eval = 0., 0
    for _ in range(cfg.eval_steps):
        b = get_batch(val_data, cfg)
        logits, _ = model(b)
        tgt = b[:, 1:]
        contextual_acc += (logits.argmax(-1) == tgt).sum().item()
        n_eval += tgt.numel()
    contextual_acc /= n_eval

    pos_embed = nn.Embedding(cfg.max_seq_len, cfg.manifold_dim).to(device)
    original_forward = model.embedding.manifold.forward

    def positional_override(x_sparse):
        B, T, D = x_sparse.shape
        pos = torch.arange(T, device=x_sparse.device)
        return pos_embed(pos).unsqueeze(0).expand(B, -1, -1)

    model.embedding.manifold.forward = positional_override
    positional_acc, n_eval = 0., 0
    for _ in range(cfg.eval_steps):
        b = get_batch(val_data, cfg)
        logits, _ = model(b)
        tgt = b[:, 1:]
        positional_acc += (logits.argmax(-1) == tgt).sum().item()
        n_eval += tgt.numel()
    positional_acc /= n_eval
    model.embedding.manifold.forward = original_forward

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    bars = axes[0].bar(['Probe\n(from q_t)', 'Baseline\n(majority)'],
                       [probe_acc, baseline_acc], color=['steelblue', 'lightcoral'])
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Test 4: Linear Probe on q_t\n(predict previous character)')
    axes[0].set_ylim(0, 1)
    for bar, val in zip(bars, [probe_acc, baseline_acc]):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{val:.1%}', ha='center', fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')

    bars = axes[1].bar(['Contextual\n(v10.1)', 'Positional\n(ablated)'],
                       [contextual_acc, positional_acc], color=['steelblue', 'lightcoral'])
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Test 5: Manifold Ablation\n(contextual vs positional at test time)')
    axes[1].set_ylim(0, max(contextual_acc, positional_acc) * 1.3 + 0.01)
    for bar, val in zip(bars, [contextual_acc, positional_acc]):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{val:.1%}', ha='center', fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.suptitle('PhD Diagnostics: Context Accumulation (Tests 4-5)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"\nTest 4: Probe accuracy = {probe_acc:.1%} (baseline: {baseline_acc:.1%})")
    print(f"Test 5: Contextual = {contextual_acc:.1%} | Positional (ablated) = {positional_acc:.1%}")
    print(f"         Drop = {contextual_acc - positional_acc:.1%}pp")

test_probing_and_ablation(model, config)

In [ ]:
@torch.no_grad()
def test_pure_sde_analysis(model, cfg):
    """Tests 6-7 (NEW): Pure SDE diagnostics — context-sensitive gradient + curvature exploration.

    These replace the v9 force-cooperation tests. With a single force, we instead ask:
    Test 6: Does the Hopfield gradient change direction when context changes?
    Test 7: Does high curvature actually produce broader exploration (more active atoms)?
    """
    model.eval()

    # === Test 6: Context-sensitive Hopfield gradient ===
    text_a = "The brave knight drew his sword and charged"
    text_b = "The quiet child drew his picture and smiled"

    T = min(cfg.seq_len, len(text_a), len(text_b))
    ids_a = torch.tensor(encode(text_a[:T]), dtype=torch.long).unsqueeze(0).to(device)
    ids_b = torch.tensor(encode(text_b[:T]), dtype=torch.long).unsqueeze(0).to(device)

    x_a, q_a = model.embedding(ids_a)
    x_b, q_b = model.embedding(ids_b)

    M_a, _ = model.blocks[0].memory_bank(q_a, x_a)
    M_b, _ = model.blocks[0].memory_bank(q_b, x_b)

    sd = cfg.subbundle_dim
    grad_a_chunks, grad_b_chunks = [], []
    x_a_flat = x_a.reshape(-1, cfg.fiber_dim)
    x_b_flat = x_b.reshape(-1, cfg.fiber_dim)

    for k in range(cfg.n_subbundles):
        xa_sub = x_a_flat[:, k*sd:(k+1)*sd]
        xb_sub = x_b_flat[:, k*sd:(k+1)*sd]

        sim_a = torch.bmm(M_a[k], xa_sub.unsqueeze(-1)).squeeze(-1)
        sim_b = torch.bmm(M_b[k], xb_sub.unsqueeze(-1)).squeeze(-1)
        w_a = F.softmax(5.0 * sim_a, dim=-1)
        w_b = F.softmax(5.0 * sim_b, dim=-1)

        grad_a_chunks.append(-torch.bmm(w_a.unsqueeze(1), M_a[k]).squeeze(1))
        grad_b_chunks.append(-torch.bmm(w_b.unsqueeze(1), M_b[k]).squeeze(1))

    grad_a = torch.cat(grad_a_chunks, dim=-1).reshape(1, T, cfg.fiber_dim)
    grad_b = torch.cat(grad_b_chunks, dim=-1).reshape(1, T, cfg.fiber_dim)

    cos_sim = F.cosine_similarity(grad_a[0], grad_b[0], dim=-1).cpu().numpy()

    sub_cos = []
    for k in range(cfg.n_subbundles):
        ga_sub = grad_a[0, :, k*sd:(k+1)*sd]
        gb_sub = grad_b[0, :, k*sd:(k+1)*sd]
        sub_cos.append(F.cosine_similarity(ga_sub, gb_sub, dim=-1).mean().item())

    # === Test 7: Curvature-dependent exploration ===
    batch = get_batch(val_data, cfg)
    x, q = model.embedding(batch)
    M_list, curvature = model.blocks[0].memory_bank(q, x)

    total_active = torch.zeros(cfg.batch_size, cfg.seq_len, device=device)
    for M_k in M_list:
        norms = M_k.reshape(cfg.batch_size, cfg.seq_len, cfg.k_wta_max, -1).norm(dim=-1)
        active = (norms > 1e-6).float().sum(dim=-1)
        total_active += active
    avg_active_per_sub = total_active / cfg.n_subbundles

    curv_flat = curvature.reshape(-1).cpu().numpy()
    active_flat = avg_active_per_sub.reshape(-1).cpu().numpy()

    # Binned relationship
    n_bins = 10
    bins = np.percentile(curv_flat[curv_flat > 0], np.linspace(0, 100, n_bins + 1))
    bin_means_k, bin_means_active = [], []
    for i in range(n_bins):
        mask = (curv_flat >= bins[i]) & (curv_flat < bins[i+1])
        if mask.sum() > 0:
            bin_means_k.append((bins[i] + bins[i+1]) / 2)
            bin_means_active.append(active_flat[mask].mean())

    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0,0].plot(cos_sim, 'purple', lw=2)
    axes[0,0].axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Context-blind (v1-v9)')
    axes[0,0].set_xlabel('Position'); axes[0,0].set_ylabel('Cosine Similarity')
    axes[0,0].set_title('Test 6: Hopfield Gradient Context Sensitivity\n(same position, different text)')
    axes[0,0].set_ylim(-0.5, 1.1); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

    axes[0,1].bar(range(cfg.n_subbundles), sub_cos, color='orchid', alpha=0.8)
    axes[0,1].axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Context-blind')
    axes[0,1].set_xlabel('Subbundle'); axes[0,1].set_ylabel('Mean Cosine Similarity')
    axes[0,1].set_title('Test 6: Per-Subbundle Gradient Divergence')
    axes[0,1].set_ylim(-0.5, 1.1); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

    axes[1,0].scatter(curv_flat, active_flat, alpha=0.05, s=5, c='teal')
    axes[1,0].set_xlabel('Manifold Curvature'); axes[1,0].set_ylabel('Active Atoms per Subbundle')
    axes[1,0].set_title('Test 7: Curvature vs Active Atoms')
    axes[1,0].grid(True, alpha=0.3)

    if bin_means_k:
        axes[1,1].plot(bin_means_k, bin_means_active, 'teal', lw=2, marker='o')
    axes[1,1].set_xlabel('Curvature Bin'); axes[1,1].set_ylabel('Mean Active Atoms')
    axes[1,1].set_title('Test 7: Binned Curvature-Activity Relationship')
    axes[1,1].grid(True, alpha=0.3)

    plt.suptitle('PhD Diagnostics: Pure SDE Analysis (Tests 6-7, NEW)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    corr = np.corrcoef(curv_flat, active_flat)[0, 1] if len(curv_flat) > 1 else 0
    print(f"\nTest 6: Mean gradient cosine = {cos_sim.mean():.4f} (1.0 = context-blind)")
    v6 = "PASS" if cos_sim.mean() < 0.95 else "FAIL"
    print(f"         {v6}")
    print(f"\nTest 7: Corr(curvature, active atoms) = {corr:.4f}")
    print(f"         k range: [{cfg.k_wta_min}, {cfg.k_wta_max}], Mean active: {active_flat.mean():.1f}")
    v7 = "PASS" if corr > 0.1 else "FAIL"
    print(f"         {v7} (positive = curvature-dependent exploration)")

test_pure_sde_analysis(model, config)

In [ ]:
@torch.no_grad()
def test_holonomy(model, cfg):
    """Test 8: Small-loop holonomy — nonzero curvature = path-dependent transport."""
    model.eval()

    text = "ROMEO:\nBut, soft! What light through yonder window breaks?"
    T = min(cfg.seq_len, len(text))
    ids_orig = torch.tensor(encode(text[:T]), dtype=torch.long).unsqueeze(0).to(device)

    holonomies = []
    swap_positions = range(2, T - 2)

    for pos in swap_positions:
        ids_swap = ids_orig.clone()
        ids_swap[0, pos], ids_swap[0, pos + 1] = ids_swap[0, pos + 1].item(), ids_swap[0, pos].item()

        _, info_orig = model(ids_orig, return_manifold=True)
        _, info_swap = model(ids_swap, return_manifold=True)

        q_orig = info_orig["manifold_coords"][0]
        q_swap = info_swap["manifold_coords"][0]

        downstream_div = (q_orig[pos+2:] - q_swap[pos+2:]).norm(dim=-1).mean().item()
        holonomies.append(downstream_div)

    holonomies = np.array(holonomies)

    # Divergence profile for a single swap
    mid = T // 2
    ids_swap = ids_orig.clone()
    ids_swap[0, mid], ids_swap[0, mid+1] = ids_swap[0, mid+1].item(), ids_swap[0, mid].item()
    _, info_orig = model(ids_orig, return_manifold=True)
    _, info_swap = model(ids_swap, return_manifold=True)
    q_orig = info_orig["manifold_coords"][0].cpu()
    q_swap = info_swap["manifold_coords"][0].cpu()
    pos_div = (q_orig - q_swap).norm(dim=-1).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(list(swap_positions), holonomies, 'b-o', ms=4)
    axes[0].set_xlabel('Swap Position')
    axes[0].set_ylabel('Mean Downstream ||dq||')
    axes[0].set_title('Test 8: Small-Loop Holonomy\n(swap adjacent tokens, measure downstream divergence)')
    axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5, label='Zero curvature (v1-v9)')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(pos_div, 'purple', lw=2)
    axes[1].axvline(x=mid, color='red', linestyle='--', alpha=0.7, label=f'Swap at pos {mid}')
    axes[1].set_xlabel('Position'); axes[1].set_ylabel('||q_orig - q_swap||')
    axes[1].set_title(f'Divergence Profile (swap at position {mid})')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle('PhD Diagnostics: Gauge Curvature (Test 8)', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"\nTest 8: Mean holonomy = {holonomies.mean():.4f}, Max = {holonomies.max():.4f}")
    verdict = "PASS" if holonomies.mean() > 0.01 else "FAIL"
    print(f"         {verdict} -- gauge curvature {'nonzero (path-dependent!)' if verdict == 'PASS' else 'zero (flat)'}")

test_holonomy(model, config)

In [ ]:
@torch.no_grad()
def test_sparse_patterns(model, cfg):
    """Tests 9-10: Sparse fiber bundle activation patterns."""
    model.eval()

    # Test 9: Activation pattern divergence under context variation
    text_a = "The quality of mercy is not strained, it droppeth as"
    text_b = "The quantity of arrows is not counted, it striketh at"
    T = min(cfg.seq_len, len(text_a), len(text_b))

    ids_a = torch.tensor(encode(text_a[:T]), dtype=torch.long).unsqueeze(0).to(device)
    ids_b = torch.tensor(encode(text_b[:T]), dtype=torch.long).unsqueeze(0).to(device)

    _, info_a = model(ids_a, return_manifold=True)
    _, info_b = model(ids_b, return_manifold=True)
    repr_a = info_a["final_repr"][0].cpu()
    repr_b = info_b["final_repr"][0].cpu()

    sd = cfg.subbundle_dim
    hamming_repr, emd_repr = [], []

    for t in range(T):
        sub_hammings, sub_emds = [], []
        for k in range(cfg.n_subbundles):
            ra_sub = repr_a[t, k*sd:(k+1)*sd]
            rb_sub = repr_b[t, k*sd:(k+1)*sd]
            ma = (ra_sub.abs() > ra_sub.abs().mean() * 0.1).float()
            mb = (rb_sub.abs() > rb_sub.abs().mean() * 0.1).float()
            sub_hammings.append((ma - mb).abs().sum().item() / sd)
            sorted_a = ra_sub.abs().sort(descending=True).values
            sorted_b = rb_sub.abs().sort(descending=True).values
            sub_emds.append((sorted_a - sorted_b).abs().sum().item())
        hamming_repr.append(np.mean(sub_hammings))
        emd_repr.append(np.mean(sub_emds))

    # Test 10: Subbundle activation entropy
    batch = get_batch(val_data, cfg)
    _, batch_info = model(batch, return_manifold=True)
    batch_repr = batch_info["final_repr"].cpu()

    subbundle_entropies = []
    for k in range(cfg.n_subbundles):
        sub_repr = batch_repr[:, :, k*sd:(k+1)*sd]
        topk = min(4, sd)
        _, top_dims = sub_repr.abs().topk(topk, dim=-1)
        patterns = top_dims.reshape(-1, topk).cpu().numpy()
        pattern_strs = [tuple(sorted(p)) for p in patterns]
        counts = Counter(pattern_strs)
        total = sum(counts.values())
        probs = np.array(list(counts.values())) / total
        entropy = -np.sum(probs * np.log(probs + 1e-10))
        subbundle_entropies.append(entropy)

    theoretical_max = np.log(math.comb(sd, min(4, sd)))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(hamming_repr, 'r-', lw=2)
    axes[0].set_xlabel('Position'); axes[0].set_ylabel('Normalized Hamming Distance')
    axes[0].set_title('Test 9a: Support Mask Divergence\n(similar texts, different contexts)')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(emd_repr, 'purple', lw=2)
    axes[1].set_xlabel('Position'); axes[1].set_ylabel('EMD (sorted activations)')
    axes[1].set_title('Test 9b: Activation Pattern Distance')
    axes[1].grid(True, alpha=0.3)

    axes[2].bar(range(cfg.n_subbundles), subbundle_entropies, color='teal', alpha=0.8)
    axes[2].axhline(y=theoretical_max, color='red', linestyle='--', alpha=0.5,
                     label=f'Max entropy = {theoretical_max:.1f}')
    axes[2].set_xlabel('Subbundle'); axes[2].set_ylabel('Pattern Entropy (nats)')
    axes[2].set_title('Test 10: Activation Pattern Entropy')
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.suptitle('PhD Diagnostics: Sparse Fiber Bundle Patterns (Tests 9-10)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"\nTest 9: Mean Hamming = {np.mean(hamming_repr):.4f}, Mean EMD = {np.mean(emd_repr):.4f}")
    print(f"Test 10: Mean subbundle entropy = {np.mean(subbundle_entropies):.2f} nats")
    print(f"          Theoretical max = {theoretical_max:.2f} nats")
    print(f"          Utilization = {np.mean(subbundle_entropies)/theoretical_max:.0%}")

test_sparse_patterns(model, config)

## Summary

### The Paradigm Shift

v10.1 does not modify v9. It implements the architecture the READMEs describe: a pure
contextual manifold where the landscape itself carries context, eliminating the need
for the four competing SDE forces.

```python
# v9 (4 forces fighting each other):
score = hopfield_grad(x, M_positional) + causal_conv(X) + inhibition(x) + attention(X)

# v10.1 (context IS the landscape):
score = hopfield_grad(x, M_contextual)
```

### Structural Differences from v9

| | v9 | v10.1 |
|---|---|---|
| Manifold | nn.Embedding (positional) | Parallel scan (contextual) |
| Forces in SDE | 4 (Hopfield, conv, inhibition, attention) | 1 (Hopfield only) |
| Memory routing | Context-blind | Curvature-aware |
| Manifold evolution | Static across blocks | Refined per block |
| Sparsity | Removed | Final-step proximal (Axiom 5) |
| Parameters | ~3.8M | ~2.5M (35% smaller) |

### PhD Diagnostic Suite Results

| Test | What It Measures | v9 Expected | v10.1 Target |
|---|---|---|---|
| 1 | Manifold divergence | 0.0 | >> 0 |
| 2 | Trajectory shape | Monotone | Curved, content-shaped |
| 3 | Atom overlap (Jaccard) | 1.0 | < 1.0 |
| 4 | Probe accuracy on q_t | Baseline | >> baseline |
| 5 | Ablation drop | 0 | Significant |
| 6 (NEW) | Gradient context sensitivity | cos = 1.0 | cos < 1.0 |
| 7 (NEW) | Curvature-activity correlation | N/A | Positive |
| 8 | Small-loop holonomy | 0 (flat) | > 0 (curved) |
| 9 | Activation pattern divergence | ~0 | > 0 |
| 10 | Subbundle entropy | Low | Higher |

### Key Question This Notebook Answers

> If the contextual manifold carries sufficient history, do we need ANY cross-position
> forces inside the Langevin loop? Or does a pure energy descent on a context-shaped
> landscape outperform the four-force Frankenstein SDE?

The answer is in the training curves above.

### Next Steps

If v10.1 outperforms v9:
- The pure SDE hypothesis is validated
- Proceed to Phase 3: Non-Abelian gauge (SU(2)), Riemannian Langevin, spectral Wilson line

If v10.1 underperforms v9:
- The contextual manifold alone is insufficient
- Selectively re-introduce forces (attention first, as it provides content-dependent retrieval)
- The diagnostic suite identifies exactly which capability is missing

---

*Built from first principles described in README_PhD_Exploration_claude_v2.md and
README_PhD_Exploration_Gemini.md. No code borrowed from v9.*